# This notebook is just to view tma samples

In [1]:
import os
import re
from tifffile import imread
import napari
from napari.settings import get_settings
settings=get_settings()
settings.application.ipy_interactive=True
import tifffile
import xml.etree.ElementTree as ET
import pandas as pd
import zarr
from itertools import cycle
import numpy as np

In [2]:
main_dir=rf'D:\thu\HTAN\images\ILC\level_2'

In [3]:
list_slide_antibodies={}
total_list_antibodies =[]
for file in os.listdir(main_dir):
    path = rf"D:\thu\HTAN\images\ILC\level_2\{file}"
    with tifffile.TiffFile(path) as tif:
        root = ET.fromstring(tif.ome_metadata)
    
    list_antibodies=[]
    for element in root.iter():
        if element.tag == "{http://www.openmicroscopy.org/Schemas/OME/2016-06}Channel": # this stays consistent with all tiff files
            list_antibodies.append(element.attrib['Name'])
            total_list_antibodies.append(element.attrib['Name'])
    list_slide_antibodies[f'{file}']=list_antibodies



PermissionError: [Errno 13] Permission denied: 'D:\\thu\\HTAN\\images\\ILC\\level_2\\QC'

In [5]:
final_list_antibodies=set(total_list_antibodies)

In [4]:
data={'file_name':[],
      'channel':[]}

for file, channels in list_slide_antibodies.items():
    data['file_name'].append(file)
    data['channel'].append(channels)



In [11]:
# CHECK PanCK for segmentation
list_file_with_panck=[]
for file in os.listdir(main_dir):
    path=os.path.join(main_dir,file)
    with tifffile.TiffFile(path) as tif:
        root = ET.fromstring(tif.ome_metadata)
        for element in root.iter():
            if element.tag.endswith('Channel'): # this stays consistent with all tiff files
                if element.attrib['Name'] == 'Pan-cytokeratin' or element.attrib['Name'] == 'panCK':
                    list_file_with_panck.append(file)

In [13]:
list_channel_name={}
for file in list_file_with_panck:
    path=os.path.join(main_dir,file)
    print(f"Processing file: {file}")
    with tifffile.TiffFile(path) as tif:
        image = tif.asarray()
        root = ET.fromstring(tif.ome_metadata)
        for element in root.iter():
            if element.tag.endswith('Channel'): # this stays consistent with all tiff files
                if element.attrib['Name'] == 'Pan-cytokeratin' or element.attrib['Name']=='panCK':
                    print(f"Found panCK in file: {file}")

Processing file: LSP12021-A7.ome.tif
Found panCK in file: LSP12021-A7.ome.tif
Processing file: LSP12021-B1.ome.tif
Found panCK in file: LSP12021-B1.ome.tif
Processing file: LSP12021-B5.ome.tif
Found panCK in file: LSP12021-B5.ome.tif
Processing file: LSP12021-B9.ome.tif
Found panCK in file: LSP12021-B9.ome.tif
Processing file: LSP12021-C3.ome.tif
Found panCK in file: LSP12021-C3.ome.tif
Processing file: LSP12021-C8.ome.tif
Found panCK in file: LSP12021-C8.ome.tif
Processing file: LSP12021-D2.ome.tif
Found panCK in file: LSP12021-D2.ome.tif
Processing file: LSP12021-F3.ome.tif
Found panCK in file: LSP12021-F3.ome.tif
Processing file: LSP12021-F7.ome.tif
Found panCK in file: LSP12021-F7.ome.tif
Processing file: LSP12021-H2.ome.tif


KeyboardInterrupt: 

In [19]:
def load_core(filepath,
              viewer:bool=False,
              show_all_channel:bool=False
              ):
   
    file_name=filepath.split("\\")[-1]
    
    print(f"Processing file: {file_name}")
    with tifffile.TiffFile(filepath) as tif:
        store = tif.aszarr()
        z = zarr.open(store, mode='r')
        root = ET.fromstring(tif.ome_metadata)
        
        channels_of_interest = ['DAPI1', 'Pan-cytokeratin', 'CD45', 'aSMA', 'CD31','panCK','DNA','Vimentin','VIM','Vim']
        if viewer:
            view_images(file_name,root, channels_of_interest)
        
        else:
            channel_data = {}
            for element in root.iter():
                if element.tag.endswith('Channel'):
                    name = element.attrib['Name']
                    for i in channels_of_interest:
                        if name == i:
                            channel = int(element.attrib['ID'].split(":")[-1])
                            channel_data[name]=channel
            return channel_data


In [18]:
def view_images(file_name, root, channels_of_interest):
    colors = ['green', 'bop blue', 'bop orange', 'bop purple', 'magenta', 'red', 'yellow']
    viewer = napari.Viewer()
    tif= tifffile.TiffFile(os.path.join(main_dir,file_name))
    store = tif.aszarr()
    z = zarr.open(store, mode='r')
    for element in root.iter():
        if element.tag.endswith('Channel'):
            name = element.attrib['Name']
            if name in channels_of_interest:
                channel = int(element.attrib['ID'].split(":")[-1])
                img = z[0][channel]
                low, high = np.percentile(img, (2.5, 99.8))
                viewer.add_image(img, name=f'{file_name}_{name}', blending='additive',
                            contrast_limits=[low, high], colormap=colors[channels_of_interest.index(name) % len(colors)])
    return viewer

In [8]:
file_cyto_markers={}
for i in os.listdir(main_dir):
    if 'QC' not in i:
        file_name=i.split("\\")[-1]
        file_path=os.path.join(main_dir, i)
        file_cyto_markers[file_name]=load_core(file_path, viewer=False)

Processing file: LSP12021-A7.ome.tif
Processing file: LSP12021-B1.ome.tif
Processing file: LSP12021-B5.ome.tif
Processing file: LSP12021-B9.ome.tif
Processing file: LSP12021-C3.ome.tif
Processing file: LSP12021-C8.ome.tif
Processing file: LSP12021-D2.ome.tif
Processing file: LSP12021-F3.ome.tif
Processing file: LSP12021-F7.ome.tif
Processing file: LSP12021-H2.ome.tif
Processing file: LSP12021-H6.ome.tif
Processing file: LSP12021-H8.ome.tif
Processing file: LSP12022-A7.ome.tif
Processing file: LSP12022-B1.ome.tif
Processing file: LSP12022-B5.ome.tif
Processing file: LSP12022-B9.ome.tif
Processing file: LSP12022-C3.ome.tif
Processing file: LSP12022-C8.ome.tif
Processing file: LSP12022-D2.ome.tif
Processing file: LSP12022-F3.ome.tif
Processing file: LSP12022-F7.ome.tif
Processing file: LSP12022-H2.ome.tif
Processing file: LSP12022-H6.ome.tif
Processing file: LSP12022-H8.ome.tif
Processing file: OHSU_TMA1_004-A7.ome.tif
Processing file: OHSU_TMA1_004-B1.ome.tif
Processing file: OHSU_TMA1_0

In [14]:
# 96 files
file_cyto_with_panck=[]
for i in file_cyto_markers.keys():
    if 'Pan-cytokeratin' in file_cyto_markers[i].keys() or 'panCK' in file_cyto_markers[i].keys():
        file_cyto_with_panck.append(i)


In [15]:
for i in  file_cyto_with_panck:
    if (len(list(file_cyto_markers[i].keys()))) > 4:
        print(i)

OHSU_TMA1_004-A7.ome.tif
OHSU_TMA1_004-B1.ome.tif
OHSU_TMA1_004-B5.ome.tif
OHSU_TMA1_004-B9.ome.tif
OHSU_TMA1_004-C3.ome.tif
OHSU_TMA1_004-C8.ome.tif
OHSU_TMA1_004-D2.ome.tif
OHSU_TMA1_004-F3.ome.tif
OHSU_TMA1_004-F7.ome.tif
OHSU_TMA1_004-H2.ome.tif
OHSU_TMA1_004-H6.ome.tif
OHSU_TMA1_004-H8.ome.tif
OHSU_TMA1_010-A7.ome.tif
OHSU_TMA1_010-B1.ome.tif
OHSU_TMA1_010-B5.ome.tif
OHSU_TMA1_010-B9.ome.tif
OHSU_TMA1_010-C3.ome.tif
OHSU_TMA1_010-C8.ome.tif
OHSU_TMA1_010-D2.ome.tif
OHSU_TMA1_010-F3.ome.tif
OHSU_TMA1_010-F7.ome.tif
OHSU_TMA1_010-H2.ome.tif
OHSU_TMA1_010-H6.ome.tif
OHSU_TMA1_010-H8.ome.tif
TNP-TMA-29_A7-ILC-3A.ome.tif
TNP-TMA-29_B1-ILC-4A.ome.tif
TNP-TMA-29_B5-ILC-2A.ome.tif
TNP-TMA-29_B9-ILC-4B.ome.tif
TNP-TMA-29_C3-LuBH-3B.ome.tif
TNP-TMA-29_C8-LuBH-3A.ome.tif
TNP-TMA-29_D2-ILC-5A.ome.tif
TNP-TMA-29_F3-ILC-3B.ome.tif
TNP-TMA-29_F7-ILC-5B.ome.tif
TNP-TMA-29_H2-ILC-1A.ome.tif
TNP-TMA-29_H6-ILC-2B.ome.tif
TNP-TMA-29_H8-ILC-1B.ome.tif


In [20]:
''' 'TNP-TMA-7-Scene-H02.ome.tif': {'DAPI1': 0,
  'aSMA': 16,
  'CD45': 23,
  'CD31': 28,
  'Vim': 31},
'''
file_name='TNP-TMA-29_A7-ILC-3A.ome.tif'
file_path=os.path.join(main_dir, file_name)
load_core(file_path, viewer=True)

Processing file: TNP-TMA-29_A7-ILC-3A.ome.tif


In [149]:
list_patient=[]
for i in list(samples_with_er_metadata['Biospecimen']):
    x=i.split('_')
    patient=('-').join(x[0:2])
    list_patient.append(patient )
set(list_patient)

{'HTA14-12', 'HTA14-16', 'HTA14-24', 'HTA14-30', 'HTA14-49', 'HTA14-7'}

##### SO all patients can check ER and HER2 status although the metadata do not have 

In [44]:
test_slide='OHSU_TMA1_004-C3.ome.tif'
test_path=os.path.join(main_dir,test_slide)
with tifffile.TiffFile(test_path) as tif:
    print(len(tif.series[0].levels))

3


In [20]:
'''Check if the biomarkers overlap with each other or not so that I can use them for composite image for segmentation'''
from itertools import combinations
from scipy.stats import pearsonr
def check_overlap(file_path, channels_to_check ):
    channels_to_check = channels_to_check 

    print(f"\n{file_path}")
    with tifffile.TiffFile(os.path.join(main_dir, file_path)) as tif:
        store = tif.aszarr()
        z = zarr.open(store, mode='r')
        root = ET.fromstring(tif.ome_metadata)
        
        channels = {}
        for element in root.iter():
            if element.tag.endswith('Channel'):
                name = element.attrib['Name']
                if name in channels_to_check:
                    idx = int(element.attrib['ID'].split(":")[-1])
                    channels[name] = z[0][idx][:].flatten().astype(float)
        

        for ch1, ch2 in combinations(channels_to_check, 2):
            if ch1 in channels and ch2 in channels:
                r, _ = pearsonr(channels[ch1], channels[ch2])
                print(f"  {ch1} vs {ch2}: r = {r:.3f}")

In [21]:
check_overlap(file_path, channels_to_check = ['Pan-cytokeratin', 'CD45', 'aSMA', 'CD31','panCK','Vimentin','VIM','Vim'])


D:\thu\HTAN\images\ILC\level_2\TNP-TMA-29_A7-ILC-3A.ome.tif
  CD45 vs CD31: r = 0.155
  CD45 vs panCK: r = 0.328
  CD45 vs VIM: r = 0.457
  CD31 vs panCK: r = 0.148
  CD31 vs VIM: r = 0.442
  panCK vs VIM: r = 0.401


# measure the nuclei size:

In [27]:
# measure the nuclei size:
import numpy as np

shapes_layer= viewer.layers['Shapes']
for i in range(len(shapes_layer.data)):
    line= shapes_layer.data[i]
    length=np.linalg.norm(line[1]-line[0])
    print(f"Length of nucleus {i}: {length}")
    avg_length= length/len(shapes_layer.data)

print(f"Average length of nuclei: {avg_length}")

Length of nucleus 0: 0.88526451587677
Length of nucleus 1: 14.862870216369629
Length of nucleus 2: 18.157041549682617
Length of nucleus 3: 0.6970077157020569
Length of nucleus 4: 17.811796188354492
Length of nucleus 5: 17.8651123046875
Length of nucleus 6: 14.677618026733398
Length of nucleus 7: 18.531328201293945
Length of nucleus 8: 17.516094207763672
Length of nucleus 9: 11.461565971374512
Length of nucleus 10: 15.25926399230957
Average length of nuclei: 1.3872058391571045


In [16]:
meta_data_ilc=pd.read_csv(rf"R:\Thu\From_2026\6. Other\DL_Summer_2026\table_data.tsv",sep="\t")
meta_data_ilc_patient=pd.read_csv(rf"R:\Thu\From_2026\6. Other\DL_Summer_2026\ilc_patient_ID_table_data.tsv",sep="\t")

In [7]:
meta_data_ilc_level_3=pd.read_csv(rf"R:\Thu\From_2026\6. Other\DL_Summer_2026\ilc_level_3_table_data.tsv",sep="\t")

In [12]:
meta_data_nst=pd.read_csv(rF"R:\Thu\From_2026\6. Other\DL_Summer_2026\nst_table_data.tsv",sep='\t')

In [66]:
nst_file=os.listdir(rf"D:\thu\HTAN\images\NST")


In [67]:
pd.Series(meta_data_nst[meta_data_nst['Filename'].str.contains('|'.join(nst_file))==False]['Synapse Id'].tolist()).to_csv('nst_file_names.txt', index=False)

In [15]:
from itables import show
show(meta_data_ilc)

Loading ITables v2.7.0 from the internet... (need help?)


In [ ]:
metadata=pd.read_csv(rf"R:\Thu\From_2026\6. Other\DL_Summer_2026\table_data.tsv",sep="\t")
metadata['Filename']=metadata['Filename'].str.split('/').str[1]
